In [1]:
import warnings
warnings.filterwarnings('ignore')
import os, re, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from numba import njit, prange
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

# Data 
DATA_START = '2000-01-01'
END_DATE = '2026-01-01'
BENCHMARK = '^SP500TR'
RF_TICKER = '^IRX' # 13-week T-bill, annualized percent
TRADING_DAYS = 252

# Rolling windows
FORMATION_MONTHS = 12 # form pairs on the first 12 months
TRADING_MONTHS = 6 # trade them on the next 6 months
STEP_MONTHS = 6 # then step forward 6 months
FIRST_FORMATION = '2000-01-01'
LAST_FORMATION = '2025-12-31'

# Pair selection 
K_OPTION = [2,4,7] # candidate cluster amounts, chosen by CVI for each window (Silhouette is the tie-breaker)
N_PAIRS = 20 # pairs traded per method

# Trading rule: following the Gatev et al. paper (2006)
ENTRY_SD = 2.0   # Open when the spread moves 2 formation-period standard deviations
LAG_DAYS = 1    # Trade the day after the signal
COST_LEVELS_BPS = [0, 5, 10, 20]

# Significance testing
N_BOOT, BLOCK_DAYS, BOOT_SEED = 5000, 20, 42

# Data Screening carried over from the first study 
KNOWN_EXCLUDE = ['FNMA', 'FMCC', 'EP', 'CCU', 'CB']

# Models 
METHODS = ['MAN', 'DTW', 'OCP', 'GATEV']
LABELS = {'MAN': 'Clustered Manhattan', 'DTW': 'Clustered DTW', 'OCP': 'Clustered OCP',
          'GATEV': 'Gatev Distance', 'Naive': 'Naive Buy-and-Hold'}
COLORS = {'MAN': '#9467bd', 'DTW': '#1f77b4', 'OCP': '#ff7f0e', 'GATEV': '#2ca02c', 'Naive': '#7f7f7f'}
PLANNED_COMPARISONS = [('MAN', 'GATEV'), ('DTW', 'MAN'), ('OCP', 'DTW')]

# Files
TAG = 'rolling_sp500'
CACHE_DIR = f'cache_{TAG}'
os.makedirs(CACHE_DIR, exist_ok=True)

print(f'Rolling design: form {FORMATION_MONTHS} months, trade {TRADING_MONTHS} months, step {STEP_MONTHS} months')
print(f'Models: {[LABELS[m] for m in METHODS]}')
print(f'Planned comparisons: {[f"{LABELS[a]} vs {LABELS[b]}" for a, b in PLANNED_COMPARISONS]}')
print(f'Cache folder: {CACHE_DIR}')

Rolling design: form 12 months, trade 6 months, step 6 months
Models: ['Clustered Manhattan', 'Clustered DTW', 'Clustered OCP', 'Gatev Distance']
Planned comparisons: ['Clustered Manhattan vs Gatev Distance', 'Clustered DTW vs Clustered Manhattan', 'Clustered OCP vs Clustered DTW']
Cache folder: cache_rolling_sp500


In [2]:
hist_url = ('https://raw.githubusercontent.com/fja05680/sp500/master/'
            'S%26P%20500%20Historical%20Components%20%26%20Changes.csv')
hist = pd.read_csv(hist_url)
hist['date'] = pd.to_datetime(hist['date'])
hist = hist.sort_values('date').reset_index(drop=True)
hist.to_csv(f'sp500_membership_{TAG}.csv', index=False)

suffix_pattern = re.compile(r'-\d{6}$')
def clean(t):
    return suffix_pattern.sub('', t.strip()).replace('.', '-')

def members_on(date):
    row = hist[hist['date'] <= date].iloc[-1]
    tickers = {clean(t) for t in row['tickers'].split(',')}
    return sorted(tickers - set(KNOWN_EXCLUDE))

windows = []
form_start = pd.Timestamp(FIRST_FORMATION)
last_end = pd.Timestamp(LAST_FORMATION)
while True:
    form_end  = form_start + pd.DateOffset(months=FORMATION_MONTHS) - pd.Timedelta(days=1)
    trade_start = form_end + pd.Timedelta(days=1)
    trade_end = trade_start + pd.DateOffset(months = TRADING_MONTHS) - pd.Timedelta(days=1)
    if trade_end > last_end:
        break
    windows.append({'window': len(windows) + 1,
                    'form_start': form_start, 'form_end': form_end,
                    'trade_start': trade_start, 'trade_end': trade_end,
                    'members': members_on(form_start)})
    form_start = form_start + pd.DateOffset(months=STEP_MONTHS)

window_df = pd.DataFrame(windows)
window_df['n_members'] = window_df['members'].apply(len)
print(window_df[['window', 'form_start', 'form_end', 'trade_start', 'trade_end', 'n_members']].to_string(index=False))

all_tickers = sorted(set().union(*window_df['members']))
print(f'\n{len(windows)} windows; {len(all_tickers)} distinct tickers across all windows')

with open(f'{CACHE_DIR}/windows.json','w') as f:
    json.dump([{k: (str(v.date()) if isinstance(v, pd.Timestamp) else v) for k, v in w.items()}
              for w in windows], f, indent=1)

 window form_start   form_end trade_start  trade_end  n_members
      1 2000-01-01 2000-12-31  2001-01-01 2001-06-30        487
      2 2000-07-01 2001-06-30  2001-07-01 2001-12-31        487
      3 2001-01-01 2001-12-31  2002-01-01 2002-06-30        487
      4 2001-07-01 2002-06-30  2002-07-01 2002-12-31        490
      5 2002-01-01 2002-12-31  2003-01-01 2003-06-30        491
      6 2002-07-01 2003-06-30  2003-07-01 2003-12-31        490
      7 2003-01-01 2003-12-31  2004-01-01 2004-06-30        489
      8 2003-07-01 2004-06-30  2004-07-01 2004-12-31        489
      9 2004-01-01 2004-12-31  2005-01-01 2005-06-30        489
     10 2004-07-01 2005-06-30  2005-07-01 2005-12-31        490
     11 2005-01-01 2005-12-31  2006-01-01 2006-06-30        490
     12 2005-07-01 2006-06-30  2006-07-01 2006-12-31        491
     13 2006-01-01 2006-12-31  2007-01-01 2007-06-30        492
     14 2006-07-01 2007-06-30  2007-07-01 2007-12-31        492
     15 2007-01-01 2007-12-31  2008-01-0

In [3]:
# Daily prices, benchmark and T-bill: load frozen files if they exist, otherwise download and save
PRICE_FILE = f'daily_prices_{TAG}.csv.gz'
BENCH_FILE = f'bench_daily_{TAG}.csv'
RF_FILE = f'rf_daily_{TAG}.csv'

def to_close_frame(raw):
    close = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw[['Close']]
    return close if isinstance(close, pd.DataFrame) else close.to_frame()

if os.path.exists(PRICE_FILE):
    prices = pd.read_csv(PRICE_FILE, index_col=0, parse_dates=True)
    print(f'Loaded frozen prices from {PRICE_FILE}')
else: 
    raw = yf.download(all_tickers, start=DATA_START, end=END_DATE, interval='1d',
                      auto_adjust=True, progress=True, threads=True)
    prices = to_close_frame(raw).dropna(how='all')   #drop days with no data
    prices = prices.dropna(axis=1, how='all')
    prices.to_csv(PRICE_FILE, float_format='%.6f')
    print(f'Downloaded and saved prices to {PRICE_FILE}')
    

_IncompleteInputError: incomplete input (4262685139.py, line 19)